# Paso 6 - Construcción y Optimización del Modelo

**Problema de negocio:** Los hoteles pierden ingresos por cancelaciones de reservas.
El 37% de las reservas se cancelan — nuestro modelo predice si una reserva se va a cancelar
antes de que ocurra para que el hotel pueda tomar medidas preventivas.

**Variable objetivo (Y):** is_canceled (0 = No cancelada, 1 = Cancelada)

**Modelos comparados:** Regresión Logística, Árbol de Decisión, Naive Bayes

> ⚠️ Ejecuta primero `03_EDA_COMPLETO_OK.ipynb` para generar los archivos necesarios
> ⚠️ El archivo `hotel_bookings.csv` debe estar en la misma carpeta

## Paso 1 - Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib                     # para guardar el modelo entrenado
import warnings
warnings.filterwarnings('ignore')

# Modelos de clasificación
from sklearn.linear_model import LogisticRegression      # modelo lineal para clasificación binaria
from sklearn.tree import DecisionTreeClassifier          # árbol de decisión
from sklearn.naive_bayes import GaussianNB               # modelo probabilístico

# Métricas de evaluación
from sklearn.metrics import accuracy_score               # % de aciertos totales
from sklearn.metrics import precision_score              # de las que predijo canceladas, cuántas lo eran
from sklearn.metrics import recall_score                 # de las que se cancelaron, cuántas detectó
from sklearn.metrics import f1_score                     # equilibrio entre precision y recall
from sklearn.metrics import roc_auc_score                # capacidad de distinguir entre clases
from sklearn.metrics import classification_report        # resumen completo de métricas
from sklearn.metrics import ConfusionMatrixDisplay       # para dibujar la matriz de confusión
from sklearn.metrics import RocCurveDisplay              # para dibujar la curva ROC

# Herramientas de optimización y validación
from sklearn.model_selection import GridSearchCV         # busca los mejores hiperparámetros
from sklearn.model_selection import cross_val_score      # validación cruzada
from sklearn.preprocessing import StandardScaler         # escalado de datos

print('Librerías importadas correctamente')

## Paso 2 - Cargar los datos del EDA

Cargamos los archivos generados en el Paso 5 (EDA).

In [ ]:
# Cargamos los 4 archivos generados en el EDA
X_train = pd.read_csv('X_train.csv')            # variables de entrenamiento (80%)
X_test  = pd.read_csv('X_test.csv')             # variables de test (20%)
y_train = pd.read_csv('y_train.csv').squeeze()  # cancelaciones de entrenamiento (0/1)
y_test  = pd.read_csv('y_test.csv').squeeze()   # cancelaciones de test (0/1)

# Cargamos la lista de features guardada en el EDA
# Este archivo tiene los nombres de las 245 variables en el orden exacto que espera el modelo
with open('features.json') as f:
    FEATURES = json.load(f)

print('Datos cargados correctamente')
print('Datos para entrenar:', X_train.shape)
print('Datos para el test: ', X_test.shape)
print('Número de features: ', len(FEATURES))

## Paso 3 - Escalado de datos

La Regresión Logística necesita que todas las variables estén en la misma escala.
El **StandardScaler** convierte cada variable para que tenga media 0 y desviación 1.

Sin esto, variables con valores grandes como `lead_time` (0-730 días)
dominarían sobre variables binarias que solo valen 0 o 1.

In [ ]:
# Creamos el escalador
scaler = StandardScaler()

# fit_transform: aprende la escala del train y lo transforma
# IMPORTANTE: solo aprendemos la escala con el train, nunca con el test
X_train_scaled = scaler.fit_transform(X_train)

# transform: aplica la misma escala aprendida al test sin re-aprender
# el test nunca debe influir en el aprendizaje del modelo
X_test_scaled = scaler.transform(X_test)

print('Escalado aplicado correctamente')

## Paso 4 - Entrenar los 3 modelos

Probamos 3 modelos con enfoques matemáticos muy distintos entre sí:

- **Regresión Logística**: modelo lineal, calcula una ecuación de probabilidad de cancelación
- **Árbol de Decisión**: toma decisiones con reglas tipo 'si lead_time > 100 y sin depósito → cancela'
- **Naive Bayes**: modelo probabilístico, calcula la probabilidad de cancelación para cada variable por separado

In [ ]:
# Modelo 1: Regresión Logística
# max_iter=1000: máximo de iteraciones para que el modelo converja
# random_state=42: semilla para que los resultados sean reproducibles
# n_jobs=-1: usa todos los núcleos del procesador para ir más rápido
print('Entrenando Regresión Logística...')
lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train_scaled, y_train)  # entrenamos con los datos de train
print('✅ Regresión Logística entrenada')

In [ ]:
# Modelo 2: Árbol de Decisión
# max_depth=5: limitamos la profundidad para evitar overfitting (memorizar el train)
# min_samples_split=20: necesita al menos 20 muestras para crear una nueva rama
# min_samples_leaf=10: cada hoja final debe tener al menos 10 muestras
print('Entrenando Árbol de Decisión...')
dt = DecisionTreeClassifier(
    max_depth=5,           # profundidad máxima del árbol
    min_samples_split=20,  # mínimo de muestras para dividir un nodo
    min_samples_leaf=10,   # mínimo de muestras en una hoja final
    random_state=42
)
dt.fit(X_train_scaled, y_train)
print('✅ Árbol de Decisión entrenado')

In [ ]:
# Modelo 3: Naive Bayes
# Es el más sencillo de entrenar — no tiene hiperparámetros complejos
# Asume que todas las variables son independientes entre sí
print('Entrenando Naive Bayes...')
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)
print('✅ Naive Bayes entrenado')

## Paso 5 - Tabla comparativa de los 3 modelos

Evaluamos cada modelo con las mismas métricas para comparar de forma justa:

- **Accuracy**: % total de predicciones correctas
- **Precision**: de las que predijo canceladas, cuántas realmente lo eran
- **Recall**: de las que realmente se cancelaron, cuántas detectó el modelo
- **F1-Score**: equilibrio entre Precision y Recall
- **ROC-AUC**: capacidad de distinguir canceladas de no canceladas (0.5=azar, 1.0=perfecto)

In [ ]:
# Diccionario con los 3 modelos
modelos = {
    'Regresión Logística': lr,
    'Árbol de Decisión':   dt,
    'Naive Bayes':         nb
}

# Lista donde guardamos los resultados de cada modelo
resultados = []

for nombre, modelo in modelos.items():
    # Predicciones sobre el test (datos que el modelo nunca ha visto)
    y_pred = modelo.predict(X_test_scaled)            # predicción: 0 o 1
    y_prob = modelo.predict_proba(X_test_scaled)[:, 1]  # probabilidad de cancelación

    # Calculamos todas las métricas y las guardamos
    resultados.append({
        'Modelo':    nombre,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1-Score':  round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4)
    })

# Convertimos la lista en tabla para verlo mejor
df_resultados = pd.DataFrame(resultados).set_index('Modelo')
print('=== COMPARATIVA DE MODELOS ===')
df_resultados

## Paso 6 - Gráfico comparativo

In [ ]:
# Gráfico de barras agrupadas con todas las métricas
df_resultados.plot(kind='bar', figsize=(12, 5), colormap='Set2', edgecolor='white')
plt.title('Comparativa de modelos de clasificación')
plt.ylabel('Score (0 a 1)')
plt.ylim(0.4, 1.0)  # empezamos en 0.4 para ver mejor las diferencias
plt.xticks(rotation=10)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('comparativa_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

## Paso 7 - Modelo ganador y justificación

In [ ]:
# Identificamos el modelo ganador por ROC-AUC (nuestra métrica principal)
ganador = df_resultados['ROC-AUC'].idxmax()
auc_ganador = df_resultados.loc[ganador, 'ROC-AUC']

print('=======================================')
print('        MODELO GANADOR')
print('=======================================')
print(f'\n🏆 {ganador}')
print(f'   ROC-AUC: {auc_ganador}')
print()
print('¿Por qué este modelo?')
print('  Tiene el mejor ROC-AUC — mide la capacidad')
print('  de distinguir reservas canceladas de no canceladas')
print('  Un ROC-AUC cercano a 1.0 significa que el modelo')
print('  casi nunca se equivoca al clasificar')
print()
print('Ahora optimizamos sus hiperparámetros con GridSearchCV')

## Paso 8 - Optimización de hiperparámetros con GridSearchCV

Los **hiperparámetros** son los ajustes del modelo que controlamos nosotros.
**GridSearchCV** prueba todas las combinaciones posibles y nos dice cuál es la mejor.
Para cada combinación hace **validación cruzada de 3 folds**:
divide el train en 3 partes, entrena con 2 y valida con 1 — repite 3 veces.

In [ ]:
# Optimización para Regresión Logística
# C controla la regularización:
# C pequeño (0.01) → modelo más simple y conservador
# C grande (100)   → modelo más complejo y ajustado
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100]  # probamos estos 5 valores de C
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    param_grid_lr,      # combinaciones a probar
    cv=3,               # validación cruzada de 3 folds
    scoring='roc_auc',  # optimizamos por ROC-AUC
    n_jobs=-1           # usa todos los núcleos del procesador
)

print('Optimizando Regresión Logística...')
grid_lr.fit(X_train_scaled, y_train)
print(f'✅ Mejor C: {grid_lr.best_params_["C"]}')
print(f'   Mejor ROC-AUC en validación: {grid_lr.best_score_:.4f}')

In [ ]:
# Optimización para Árbol de Decisión
# max_depth:         profundidad máxima del árbol
# min_samples_split: mínimo de muestras para dividir un nodo
# min_samples_leaf:  mínimo de muestras en una hoja final
param_grid_dt = {
    'max_depth':         [3, 5, 7, 10],   # profundidades a probar
    'min_samples_split': [10, 20, 50],    # mínimo para dividir
    'min_samples_leaf':  [5, 10, 20]      # mínimo en hoja
}

grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_dt,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1
)

print('Optimizando Árbol de Decisión...')
grid_dt.fit(X_train_scaled, y_train)
print(f'✅ Mejores parámetros: {grid_dt.best_params_}')
print(f'   Mejor ROC-AUC en validación: {grid_dt.best_score_:.4f}')

## Paso 9 - Evaluación final del modelo ganador

In [ ]:
# Evaluamos los modelos optimizados sobre el test
modelos_opt = {
    'Regresión Logística (opt)': grid_lr.best_estimator_,
    'Árbol de Decisión (opt)':   grid_dt.best_estimator_,
    'Naive Bayes':               nb  # Naive Bayes no tiene hiperparámetros clave
}

resultados_opt = []

for nombre, modelo in modelos_opt.items():
    y_pred = modelo.predict(X_test_scaled)
    y_prob = modelo.predict_proba(X_test_scaled)[:, 1]
    resultados_opt.append({
        'Modelo':    nombre,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1-Score':  round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4)
    })

df_opt = pd.DataFrame(resultados_opt).set_index('Modelo')
print('=== MODELOS OPTIMIZADOS ===')
df_opt

In [ ]:
# Seleccionamos el modelo final ganador por ROC-AUC
ganador_final = df_opt['ROC-AUC'].idxmax()
best_model = modelos_opt[ganador_final]

# Predicciones finales con el modelo ganador
y_pred_final = best_model.predict(X_test_scaled)
y_prob_final = best_model.predict_proba(X_test_scaled)[:, 1]

print(f'🏆 Modelo final: {ganador_final}')
print()
print('=== MÉTRICAS FINALES ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}  → % total de aciertos')
print(f'Precision: {precision_score(y_test, y_pred_final):.4f}  → de las que predijo canceladas, cuántas lo eran')
print(f'Recall:    {recall_score(y_test, y_pred_final):.4f}  → de las que se cancelaron, cuántas detectó')
print(f'F1-Score:  {f1_score(y_test, y_pred_final):.4f}  → equilibrio entre precision y recall')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_final):.4f}  → capacidad de discriminación')
print()
print(classification_report(y_test, y_pred_final, target_names=['No cancelada', 'Cancelada']))

## Paso 10 - Matriz de confusión y curva ROC

**Matriz de confusión** — los 4 tipos de predicción posibles:
- **Verdadero Negativo**: predijo no cancelada y era correcta ✅
- **Falso Positivo**: predijo cancelada pero no lo era ❌
- **Falso Negativo**: predijo no cancelada pero sí se canceló ❌
- **Verdadero Positivo**: predijo cancelada y era correcta ✅

**Curva ROC** — cuanto más arriba y a la izquierda, mejor el modelo.
La línea diagonal punteada representa un clasificador aleatorio (ROC-AUC = 0.5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_final,
    display_labels=['No cancelada', 'Cancelada'],
    cmap='Blues',   # azules más oscuros = más predicciones
    ax=axes[0]
)
axes[0].set_title(f'Matriz de confusión — {ganador_final}')

# Curva ROC
RocCurveDisplay.from_predictions(
    y_test,
    y_prob_final,   # usamos probabilidades, no predicciones binarias
    ax=axes[1],
    color='#e74c3c'
)
# Línea diagonal = clasificador aleatorio (lo que queremos superar)
axes[1].plot([0,1], [0,1], 'k--', alpha=0.5, label='Clasificador aleatorio (AUC=0.5)')
axes[1].set_title(f'Curva ROC — {ganador_final}')
axes[1].legend()

plt.tight_layout()
plt.savefig('matriz_roc.png', dpi=150, bbox_inches='tight')
plt.show()

## Paso 11 - Variables más importantes

Analizamos qué variables influyeron más en las predicciones del modelo ganador.

In [ ]:
# Extraemos la importancia según el tipo de modelo ganador
if hasattr(best_model, 'coef_'):
    # Regresión Logística: usamos los coeficientes
    # Positivo → aumenta probabilidad de cancelación
    # Negativo → reduce probabilidad de cancelación
    importancias = pd.Series(best_model.coef_[0], index=FEATURES)
    top20 = importancias.abs().nlargest(20).index
    valores = importancias[top20].sort_values()
    colores = ['#e74c3c' if v > 0 else '#2ecc71' for v in valores]
    titulo = 'Coeficientes (rojo = aumenta cancelación | verde = reduce cancelación)'

elif hasattr(best_model, 'feature_importances_'):
    # Árbol de Decisión: usamos la importancia de cada variable
    importancias = pd.Series(best_model.feature_importances_, index=FEATURES)
    valores = importancias.nlargest(20).sort_values()
    colores = '#3498db'
    titulo = 'Top 20 variables más importantes — Árbol de Decisión'

plt.figure(figsize=(10, 7))
valores.plot(kind='barh', color=colores, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title(titulo)
plt.xlabel('Importancia')
plt.tight_layout()
plt.savefig('variables_importantes.png', dpi=150, bbox_inches='tight')
plt.show()

## Paso 12 - Validación cruzada

Divide el train en 5 partes. En cada ronda entrena con 4 y evalúa con 1.
Si los 5 resultados son similares, el modelo es estable y no hace **overfitting**.

**Overfitting** = el modelo memoriza el train pero falla con datos nuevos

In [ ]:
# Validación cruzada de 5 folds sobre los datos de entrenamiento
cv_scores = cross_val_score(
    best_model,       # el modelo ganador optimizado
    X_train_scaled,   # datos de entrenamiento escalados
    y_train,          # etiquetas de entrenamiento
    cv=5,             # 5 folds
    scoring='roc_auc',# métrica a evaluar
    n_jobs=-1         # paralelizar
)

print('Validación cruzada — 5 folds (ROC-AUC):')
for i, score in enumerate(cv_scores, 1):
    print(f'   Fold {i}: {score:.4f}')

print(f'\n   Media:        {cv_scores.mean():.4f}')
print(f'   Desv. std:    {cv_scores.std():.4f}  (cuanto menor, más estable el modelo)')
print(f'   ROC-AUC test: {roc_auc_score(y_test, y_prob_final):.4f}')
print()
print('→ Si media ≈ ROC-AUC test, el modelo generaliza bien (no hay overfitting)')

## Paso 13 - Guardar el modelo final

Guardamos el modelo y el escalador en archivos `.pkl`.
La app web (Paso 7) los cargará para hacer predicciones sin reentrenar.

In [ ]:
# Guardamos el modelo entrenado
# joblib.dump es como hacer una foto del modelo para usarlo después
joblib.dump(best_model, 'modelo_cancelacion.pkl')

# Guardamos el escalador
# Los datos nuevos también hay que escalarlos antes de predecir
joblib.dump(scaler, 'scaler.pkl')

print('✅ Archivos guardados:')
print('   modelo_cancelacion.pkl → el modelo entrenado')
print('   scaler.pkl             → el escalador de datos')
print('   features.json          → lista de variables (generado en el EDA)')

## Paso 14 - Resumen final para la presentación

In [ ]:
print('================================================')
print('          RESUMEN FINAL DEL PROYECTO')
print('================================================')
print()
print('PROBLEMA DE NEGOCIO:')
print('  El 37% de las reservas hoteleras se cancelan.')
print('  Nuestro modelo predice qué reservas tienen riesgo')
print('  de cancelarse para que el hotel actúe con antelación.')
print()
print('DATASET:')
print(f'  {X_train.shape[0] + X_test.shape[0]:,} reservas totales')
print(f'  {len(FEATURES)} variables predictoras (tras OneHotEncoder)')
print(f'  Train: {X_train.shape[0]:,} reservas | Test: {X_test.shape[0]:,} reservas')
print()
print('MODELOS PROBADOS:')
print('  1. Regresión Logística → modelo lineal')
print('  2. Árbol de Decisión   → modelo basado en reglas')
print('  3. Naive Bayes         → modelo probabilístico')
print()
print('OPTIMIZACIÓN:')
print('  GridSearchCV con validación cruzada de 3 folds')
print(f'  Regresión Logística → mejor C: {grid_lr.best_params_["C"]}')
print(f'  Árbol de Decisión   → {grid_dt.best_params_}')
print()
print(f'MODELO GANADOR: {ganador_final}')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred_final):.4f}')
print(f'  Recall:    {recall_score(y_test, y_pred_final):.4f}')
print(f'  F1-Score:  {f1_score(y_test, y_pred_final):.4f}')
print(f'  ROC-AUC:   {roc_auc_score(y_test, y_prob_final):.4f}')
print()
print('PATRONES CLAVE ENCONTRADOS:')
print('  ✅ El 37% de las reservas se cancelan')
print('  ✅ Más antelación = más probabilidad de cancelar')
print('     (144 días de media vs 80 días en no canceladas)')
print('  ✅ City Hotel cancela más que Resort Hotel (41% vs 28%)')
print('  ✅ Sin depósito = mayor tasa de cancelación')
print('  ✅ Agencias online generan más cancelaciones que reservas directas')
print('  ✅ Quien canceló antes tiene más probabilidad de volver a cancelar')
print()
print('MEJORAS FUTURAS:')
print('  → Clasificar clientes por nivel de riesgo (bajo/medio/alto)')
print('  → Aplicar tarifas de cancelación personalizadas según el riesgo')
print('  → Añadir modelo de predicción de precio óptimo por noche')
print('  → Reentrenar el modelo mensualmente con datos nuevos')
print('  → Añadir datos externos: eventos locales, clima, competencia')
print('================================================')